## Carico il BRT

In [1]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root.resolve())

Project root: /home/alessio/TESI/hj_reachability


In [2]:
# Percorso del BRT salvato localmente
data_path = project_root / "results" / "brt_euclidean.npz"

if not data_path.exists():
    raise FileNotFoundError(
        f"File non trovato:\n{data_path.resolve()}"
    )

data = np.load(data_path)

print("File caricato:", data_path.resolve())
print("Variabili disponibili:")
for name in data.files:
    print(f"  - {name}")

File caricato: /home/alessio/TESI/hj_reachability/results/brt_euclidean.npz
Variabili disponibili:
  - BRT
  - gradienti
  - x_rel
  - y_rel
  - theta_rel
  - v_H
  - delta_E
  - v_E
  - target_time
  - periodic_dims


## Estrarre i dati

In [3]:
BRT = data["BRT"]
gradienti = data["gradienti"]

x_rel_grid = data["x_rel"]
y_rel_grid = data["y_rel"]
theta_rel_grid = data["theta_rel"]
v_H_grid = data["v_H"]
delta_E_grid = data["delta_E"]
v_E_grid = data["v_E"]

target_time = float(data["target_time"])
periodic_dims = data["periodic_dims"]

coordinate_vectors = (
    x_rel_grid,
    y_rel_grid,
    theta_rel_grid,
    v_H_grid,
    delta_E_grid,
    v_E_grid,
)

In [4]:
grid_shape = tuple(len(vector) for vector in coordinate_vectors)

print("Forma ricostruita della griglia:", grid_shape)
print("Forma BRT:", BRT.shape)
print("Forma gradienti:", gradienti.shape)
print("Orizzonte BRT:", target_time, "s")
print("Dimensioni periodiche:", periodic_dims)

assert BRT.shape == grid_shape
assert gradienti.shape == (*grid_shape, 6)

print("\nCaricamento completato correttamente.")

Forma ricostruita della griglia: (26, 13, 15, 11, 11, 11)
Forma BRT: (26, 13, 15, 11, 11, 11)
Forma gradienti: (26, 13, 15, 11, 11, 11, 6)
Orizzonte BRT: -3.0 s
Dimensioni periodiche: [2]

Caricamento completato correttamente.


## Controllo numerico

In [5]:
n_nan_BRT = np.isnan(BRT).sum()
n_inf_BRT = np.isinf(BRT).sum()

n_nan_gradienti = np.isnan(gradienti).sum()
n_inf_gradienti = np.isinf(gradienti).sum()

print("BRT")
print("  NaN:", n_nan_BRT)
print("  Inf:", n_inf_BRT)

print("\nGradienti")
print("  NaN:", n_nan_gradienti)
print("  Inf:", n_inf_gradienti)

assert n_nan_BRT == 0
assert n_inf_BRT == 0
assert n_nan_gradienti == 0
assert n_inf_gradienti == 0

print("\nBRT e gradienti contengono soltanto valori finiti.")

BRT
  NaN: 0
  Inf: 0

Gradienti
  NaN: 0
  Inf: 0

BRT e gradienti contengono soltanto valori finiti.


In [7]:
# Controlli sul BRT

print(f"Minimo BRT: {BRT.min():.6f}")
print(f"Massimo BRT: {BRT.max():.6f}")

n_inside = np.count_nonzero(BRT <= 0)
n_outside = np.count_nonzero(BRT > 0)
n_total = BRT.size

print("\nDistribuzione del segno")
print(f"  BRT <= 0: {n_inside:,} punti ({100 * n_inside / n_total:.2f}%)")
print(f"  BRT > 0:  {n_outside:,} punti ({100 * n_outside / n_total:.2f}%)")

Minimo BRT: -23.305016
Massimo BRT: 13.482062

Distribuzione del segno
  BRT <= 0: 2,305,629 punti (34.17%)
  BRT > 0:  4,442,541 punti (65.83%)


In [8]:
# Controlli sul gradiente

gradient_names = (
    "dV/dx_rel",
    "dV/dy_rel",
    "dV/dtheta_rel",
    "dV/dv_H",
    "dV/ddelta_E",
    "dV/dv_E",
)

for dimension, name in enumerate(gradient_names):
    component = gradienti[..., dimension]

    print(f"{name}")
    print(f"  minimo:  {component.min(): .6f}")
    print(f"  massimo: {component.max(): .6f}")
    print()

dV/dx_rel
  minimo:  -9.917418
  massimo:  12.389023

dV/dy_rel
  minimo:  -12.057384
  massimo:  10.449586

dV/dtheta_rel
  minimo:  -19.598701
  massimo:  21.568453

dV/dv_H
  minimo:  -2.603590
  massimo:  6.920338

dV/ddelta_E
  minimo:  -78.786095
  massimo:  77.436363

dV/dv_E
  minimo:  -2.204033
  massimo:  12.624368



In [9]:
# Controlli sulla griglia

coordinate_names = (
    "x_rel",
    "y_rel",
    "theta_rel",
    "v_H",
    "delta_E",
    "v_E",
)

for name, vector in zip(coordinate_names, coordinate_vectors):
    steps = np.diff(vector)

    print(name)
    print(f"  punti:   {len(vector)}")
    print(f"  minimo:  {vector.min():.6f}")
    print(f"  massimo: {vector.max():.6f}")
    print(f"  passo minimo: {steps.min():.6f}")
    print(f"  passo massimo: {steps.max():.6f}")
    print()

x_rel
  punti:   26
  minimo:  -8.000000
  massimo: 17.000000
  passo minimo: 1.000000
  passo massimo: 1.000000

y_rel
  punti:   13
  minimo:  -6.000000
  massimo: 6.000000
  passo minimo: 1.000000
  passo massimo: 1.000000

theta_rel
  punti:   15
  minimo:  -0.785398
  massimo: 0.680679
  passo minimo: 0.104720
  passo massimo: 0.104720

v_H
  punti:   11
  minimo:  1.000000
  massimo: 11.000000
  passo minimo: 1.000000
  passo massimo: 1.000000

delta_E
  punti:   11
  minimo:  -0.261799
  massimo: 0.261799
  passo minimo: 0.052360
  passo massimo: 0.052360

v_E
  punti:   11
  minimo:  1.000000
  massimo: 11.000000
  passo minimo: 1.000000
  passo massimo: 1.000000



## Interpolatore BRT

In [10]:
from scipy.interpolate import RegularGridInterpolator

In [11]:
# Interpolatore del BRT

BRT_interpolator = RegularGridInterpolator(
    points=coordinate_vectors,
    values=BRT,
    method="linear",
    bounds_error=True,
)

print("Interpolatore BRT costruito correttamente.")

Interpolatore BRT costruito correttamente.


In [14]:
def evaluate_BRT(state):
    """
    Valuta il BRT nello stato relativo 6D.

    Ordine:
    [x_rel, y_rel, theta_rel, v_H, delta_E, v_E]
    """
    state = np.asarray(state, dtype=float)

    if state.shape != (6,):
        raise ValueError(
            f"Lo stato deve avere forma (6,), ricevuta {state.shape}."
        )

    interpolated_value = BRT_interpolator(state)

    return np.asarray(interpolated_value).item()

In [15]:
# Indici di un nodo circa centrale nella griglia

central_indices = tuple(
    len(vector) // 2
    for vector in coordinate_vectors
)

central_state = np.array(
    [
        vector[index]
        for vector, index in zip(
            coordinate_vectors,
            central_indices,
        )
    ]
)

interpolated_value = evaluate_BRT(central_state)
stored_value = float(BRT[central_indices])

print("Indici:", central_indices)
print("Stato centrale:", central_state)
print("Valore salvato:     ", stored_value)
print("Valore interpolato: ", interpolated_value)
print("Errore assoluto:    ", abs(interpolated_value - stored_value))

Indici: (13, 6, 7, 5, 5, 5)
Stato centrale: [ 5.          0.         -0.05235982  6.          0.          6.        ]
Valore salvato:      -1.0378766059875488
Valore interpolato:  -1.0378766059875488
Errore assoluto:     0.0


## Interpolatore gradiente

In [16]:
# Interpolatore del gradiente del BRT

gradient_interpolator = RegularGridInterpolator(
    points=coordinate_vectors,
    values=gradienti,
    method="linear",
    bounds_error=True,
)

print("Interpolatore del gradiente costruito correttamente.")

Interpolatore del gradiente costruito correttamente.


In [17]:
def evaluate_gradient(state):
    """
    Valuta il gradiente 6D del BRT nello stato fornito.

    Ordine dello stato:
    [x_rel, y_rel, theta_rel, v_H, delta_E, v_E]
    """
    state = np.asarray(state, dtype=float)

    if state.shape != (6,):
        raise ValueError(
            f"Lo stato deve avere forma (6,), ricevuta {state.shape}."
        )

    gradient = np.asarray(
        gradient_interpolator(state),
        dtype=float,
    )

    # L'interpolatore può restituire forma (1, 6)
    return gradient.reshape(6)

In [18]:
interpolated_gradient = evaluate_gradient(central_state)
stored_gradient = np.asarray(
    gradienti[central_indices],
    dtype=float,
)

print("Stato centrale:")
print(central_state)

print("\nGradiente salvato:")
print(stored_gradient)

print("\nGradiente interpolato:")
print(interpolated_gradient)

print("\nErrore assoluto per componente:")
print(np.abs(interpolated_gradient - stored_gradient))

Stato centrale:
[ 5.          0.         -0.05235982  6.          0.          6.        ]

Gradiente salvato:
[ 1.38487029 -0.06282669 -0.15580286  0.7445389   1.15731454 -0.6764046 ]

Gradiente interpolato:
[ 1.38487029 -0.06282669 -0.15580286  0.7445389   1.15731454 -0.6764046 ]

Errore assoluto per componente:
[0. 0. 0. 0. 0. 0.]


In [19]:
for name, value in zip(
    gradient_names,
    interpolated_gradient,
):
    print(f"{name:16s} = {value: .6f}")

dV/dx_rel        =  1.384870
dV/dy_rel        = -0.062827
dV/dtheta_rel    = -0.155803
dV/dv_H          =  0.744539
dV/ddelta_E      =  1.157315
dV/dv_E          = -0.676405


## Dinamica

In [20]:
import jax.numpy as jnp

from hj_reachability.systems.relative_vehicle_6d import RelativeVehicle6D

In [21]:
dynamics = RelativeVehicle6D(
    lf=1.2,
    lr=1.5,
)

print("Dinamica RelativeVehicle6D costruita correttamente.")

Dinamica RelativeVehicle6D costruita correttamente.


In [22]:
print("Limiti controllo Ego:")
print("  minimo:", np.asarray(dynamics.control_space.lo))
print("  massimo:", np.asarray(dynamics.control_space.hi))

print("\nLimiti disturbo Human:")
print("  minimo:", np.asarray(dynamics.disturbance_space.lo))
print("  massimo:", np.asarray(dynamics.disturbance_space.hi))

Limiti controllo Ego:
  minimo: [-0.087 -7.   ]
  massimo: [0.087 2.5  ]

Limiti disturbo Human:
  minimo: [-1. -7.]
  massimo: [1.  2.5]


In [23]:
test_state = jnp.array([
    5.0,                 # x_rel [m]
    1.0,                 # y_rel [m]
    np.deg2rad(10.0),    # theta_rel [rad]
    12.0,                # v_H [m/s]
    0.05,                # delta_E [rad]
    14.0,                # v_E [m/s]
])

test_time = 0.0

In [24]:
open_loop = dynamics.open_loop_dynamics(
    test_state,
    test_time,
)

control_jacobian = dynamics.control_jacobian(
    test_state,
    test_time,
)

disturbance_jacobian = dynamics.disturbance_jacobian(
    test_state,
    test_time,
)

print("Dinamica libera f(x):")
print(np.asarray(open_loop))

print("\nMatrice di controllo G(x):")
print(np.asarray(control_jacobian))

print("\nMatrice di disturbo H(x):")
print(np.asarray(disturbance_jacobian))

Dinamica libera f(x):
[-1.9175246  0.3978386 -0.2593753  0.         0.         0.       ]

Matrice di controllo G(x):
[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [1. 0.]
 [0. 1.]]

Matrice di disturbo H(x):
[[0. 0.]
 [0. 0.]
 [1. 0.]
 [0. 1.]
 [0. 0.]
 [0. 0.]]


In [25]:
assert open_loop.shape == (6,)
assert control_jacobian.shape == (6, 2)
assert disturbance_jacobian.shape == (6, 2)

print("\nLe dimensioni della dinamica sono corrette.")


Le dimensioni della dinamica sono corrette.


## Ricerca di uno stato iniziale libero